In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr


# -----------------------
# 0-1 Test Implementation (No changes needed in this function)
# -----------------------
def z1test(x, plot_pq=False, save_path=None, stock_name="", method="", level=""):
    """
    Performs the 0-1 test for chaos on a time series 'x'.
    Generates and saves plots if requested.
    """
    x = np.array(x)
    if x.ndim > 1:
        x = x.flatten()
    N = len(x)
    j = np.arange(N)
    # Use a fixed range of c values for reproducibility
    c = np.linspace(np.pi / 5, 4 * np.pi / 5, 100)
    kcorr = np.zeros(100)
    p = np.zeros((N, 100))
    q = np.zeros((N, 100))

    for its in range(100):
        p[:, its] = np.cumsum(x * np.cos(j * c[its]))
        q[:, its] = np.cumsum(x * np.sin(j * c[its]))
        M = np.zeros(int(N / 10))
        t = np.arange(1, int(N / 10) + 1)
        for n_idx, n in enumerate(t):
            M[n_idx] = np.mean((p[n:N, its] - p[:N - n, its])**2 +
                               (q[n:N, its] - q[:N - n, its])**2) - \
                       np.mean(x)**2 * (1 - np.cos(n * c[its])) / (1 - np.cos(c[its]))
        # Use np.corrcoef for direct calculation on numpy arrays, handles potential NaNs
        corr_matrix = np.corrcoef(t, M)
        kcorr[its] = corr_matrix[0, 1] if not np.isnan(corr_matrix[0, 1]) else 0.0

    k_val = np.median(kcorr)

    # Save plots if requested
    if plot_pq and save_path:
        # p-q trajectories for the first 5 c values
        plt.figure(figsize=(8, 6))
        for i in range(5):
            plt.plot(p[:, i], q[:, i], linewidth=0.7, label=f'c value {i+1}')
        plt.xlabel('p(n)')
        plt.ylabel('q(n)')
        plt.title(f'{stock_name} | Filter: {method} (L{level}) | p-q Trajectories')
        plt.grid(True)
        plt.legend(fontsize='small')
        plt.tight_layout()
        plt.savefig(os.path.join(save_path, f"{stock_name}_{method}_L{level}_pq.png"))
        plt.close()

        # Time series plot
        plt.figure(figsize=(10, 4))
        plt.plot(x, 'r-', lw=0.6)
        plt.xlabel('Time (subsampled)')
        plt.ylabel('Log Return Value')
        plt.title(f'{stock_name} | Filter: {method} (L{level}) | Cleaned Time Series')
        plt.tight_layout()
        plt.savefig(os.path.join(save_path, f"{stock_name}_{method}_L{level}_timeseries.png"))
        plt.close()

    return k_val


# -----------------------
# MODIFIED Batch Processing Script
# -----------------------
def process_all_stocks_with_all_filters(input_dir="clean_data",
                                        results_dir="0-1_results_all_filters",
                                        output_csv="0-1_results_all_filters.csv"):
    """
    Runs the 0-1 test for a list of filters on ALL stocks in the input directory.
    Plots are ONLY generated for the 'AAPL' stock.
    """
    # --- Configuration ---
    # Define the wavelet filters and levels to test independently on each stock
    filters_to_test = [
        ('haar', 4),
        ('coif5', 4),
        ('bior6.8', 4),
        ('sym8', 4)
    ]

    if not os.path.exists(results_dir):
        os.makedirs(results_dir)

    results = []
    
    # --- Loop through all files in the input directory ---
    files_to_process = [f for f in os.listdir(input_dir) if f.endswith("_clean.csv")]
    
    if not files_to_process:
        print(f"No '*_clean.csv' files found in '{input_dir}'. Exiting.")
        return
        
    print(f"Found {len(files_to_process)} stocks to process.")

    for file in files_to_process:
        stock_name = file.replace("_clean.csv", "")
        filepath = os.path.join(input_dir, file)
        
        print(f"\n--- Processing Stock: {stock_name} ---")

        try:
            df = pd.read_csv(filepath)
            if "Log_Return_Clean" not in df.columns:
                print(f"  > Warning: Missing 'Log_Return_Clean' column, skipping.")
                continue
            data = df["Log_Return_Clean"].dropna().values
            sampled_data = data[::10]  # Subsample
        except Exception as e:
            print(f"  > Error loading data for {stock_name}: {e}")
            continue

        # --- Loop through the list of filters for the current stock ---
        for method, level in filters_to_test:
            # Conditional plotting: only generate plots if the stock is AAPL
            generate_plots = (stock_name == "AAPL")
            
            if generate_plots:
                print(f"  > Testing filter '{method}' (L{level}) and generating plots...")
            else:
                print(f"  > Testing filter '{method}' (L{level})...")

            try:
                k_val = z1test(sampled_data, plot_pq=generate_plots, save_path=results_dir,
                               stock_name=stock_name, method=method, level=level)
                results.append([stock_name, method, level, k_val])
                print(f"    > K-value = {k_val:.4f}")
            except Exception as e:
                print(f"    > Error during 0-1 test: {e}")

    # --- Save final results table ---
    if not results:
        print("\nAnalysis complete, but no results were generated.")
        return
        
    results_df = pd.DataFrame(results, columns=["Stock", "Wavelet", "Level", "K"])
    results_filepath = os.path.join(results_dir, output_csv)
    results_df.to_csv(results_filepath, index=False)

    print(f"\n--- Analysis Complete ---")
    print(f"Saved comprehensive results table to: {results_filepath}")
    if any(res[0] == 'AAPL' for res in results):
        print(f"Plots for AAPL saved in the '{results_dir}' directory.")
    print("\nFinal Results Preview:")
    print(results_df.head().to_string(index=False))


# -----------------------
# Run
# -----------------------
if __name__ == "__main__":
    process_all_stocks_with_all_filters()

Found 18 stocks to process.

--- Processing Stock: AMZN ---
  > Testing filter 'haar' (L4)...
    > K-value = 0.9958
  > Testing filter 'coif5' (L4)...
    > K-value = 0.9958
  > Testing filter 'bior6.8' (L4)...
    > K-value = 0.9958
  > Testing filter 'sym8' (L4)...
    > K-value = 0.9958

--- Processing Stock: KLAC ---
  > Testing filter 'haar' (L4)...
    > K-value = 0.9978
  > Testing filter 'coif5' (L4)...
    > K-value = 0.9978
  > Testing filter 'bior6.8' (L4)...
    > K-value = 0.9978
  > Testing filter 'sym8' (L4)...
    > K-value = 0.9978

--- Processing Stock: AAPL ---
  > Testing filter 'haar' (L4) and generating plots...
    > K-value = 0.9975
  > Testing filter 'coif5' (L4) and generating plots...
    > K-value = 0.9975
  > Testing filter 'bior6.8' (L4) and generating plots...
    > K-value = 0.9975
  > Testing filter 'sym8' (L4) and generating plots...
    > K-value = 0.9975

--- Processing Stock: CSCO ---
  > Testing filter 'haar' (L4)...
    > K-value = 0.9963
  > Tes